# Detecção de Objetos em Moda — YOLOv8 vs Faster R-CNN vs SSD

Notebook de orquestração do treino e avaliação no Google Colab (GPU). O código em si
(`src/data.py`, `models.py`, `train.py`, `eval.py`, `plotting.py`) já está pronto e testado
no repositório — este notebook só chama esses scripts na ordem certa e documenta o experimento.

**Como usar este notebook:** blocos de markdown marcados com `> INSTRUÇÃO` são placeholders
para você escrever a análise/discussão pedida no enunciado do trabalho. Escreva seu texto
no lugar do placeholder e apague a linha `> INSTRUÇÃO (...)` — as células de código já estão
prontas pra rodar como estão (ajuste só os hiperparâmetros marcados com `# AJUSTE AQUI`).

## 0. Setup

In [ ]:
# Clona o repositório e instala as dependências
!git clone https://github.com/caique-veiga/deteccao-objetos-fashionpedia.git
%cd deteccao-objetos-fashionpedia
!pip install -r requirements.txt -q

In [ ]:
# Recria o .env (não vem do GitHub, de propósito). O HF_TOKEN é opcional pro Fashionpedia
# (dataset público), mas evita avisos de rate-limit e é necessário se você usar outro dataset
# do HuggingFace que exija autenticação.
%%writefile .env
HF_TOKEN=seu_token_aqui

In [ ]:
# Monta o Google Drive e aponta checkpoints/ e logs/ pra lá, via symlink.
# Por quê: o /content do Colab é apagado quando a sessão desconecta. Sem isso, um
# treino longo que cair no meio perde tudo. data/ fica local (mais rápido pra treinar;
# se a sessão cair, é só rodar a célula de conversão de novo).
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/deteccao-objetos-fashionpedia'  # AJUSTE AQUI se quiser outro caminho
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)

!rm -rf checkpoints logs
!ln -s {DRIVE_DIR}/checkpoints checkpoints
!ln -s {DRIVE_DIR}/logs logs

> INSTRUÇÃO (apague depois de rodar): confirme que a GPU está ativa em
> Ambiente de execução > Alterar tipo de ambiente de execução > GPU, antes de continuar.

In [ ]:
!nvidia-smi

## 1. Problema

> INSTRUÇÃO (apague esta linha depois de escrever): descreva em 2-3 parágrafos o problema
> de detecção de objetos em imagens de moda — por que é um problema relevante (ex: catálogo
> automático de e-commerce, tagging de peças de roupa), e por que faz sentido comparar um
> modelo single-stage (YOLOv8) com dois modelos two-stage/anchor-based (Faster R-CNN, SSD)
> em termos de trade-off entre velocidade e precisão.

## 2. Base de dados

> INSTRUÇÃO (apague esta linha depois de escrever): descreva o Fashionpedia (46 categorias,
> ~45 mil imagens de treino, formato original das anotações). Rode a célula de conversão/EDA
> abaixo primeiro — ela gera um gráfico de distribuição de classes em `logs/` que você pode
> inserir aqui como evidência.

In [ ]:
# Baixa o Fashionpedia inteiro e converte pra COCO + YOLO (pode demorar — é o dataset completo).
# Gera também o gráfico de distribuição de classes em logs/eda_class_distribution_train.png
!python src/data.py --output-dir data --eda

> INSTRUÇÃO (apague depois): exiba o gráfico de EDA e comente a distribuição de classes
> (ex: classes desbalanceadas? alguma categoria dominante?).

In [ ]:
from IPython.display import Image as IPImage
IPImage('logs/eda_class_distribution_train.png')

## 3. Metodologia

> INSTRUÇÃO (apague depois de escrever): resuma os 3 modelos e as decisões de comparação
> justa já documentadas em `src/train.py` — batch size, número de épocas e critério de
> early stopping (`--patience`) são iguais nos três; Faster R-CNN/SSD usam SGD com
> decaimento cosseno de LR (o YOLO já decai o LR automaticamente via ultralytics). Mencione
> também a limitação conhecida: data augmentation NÃO foi equalizada (YOLO usa mosaic/flip/HSV
> jitter por padrão do ultralytics; Faster R-CNN/SSD não aplicam nenhuma).

## 4. Experimentos

> INSTRUÇÃO (apague depois de escrever): antes de rodar, ajuste `EPOCHS`, `BATCH_SIZE` e
> `PATIENCE` na célula abaixo conforme o tempo de GPU disponível (comece com poucas épocas
> pra ter uma estimativa de tempo por época, depois ajuste). Depois de rodar, descreva aqui
> quantas épocas/tempo cada modelo levou e qualquer ajuste que você precisou fazer.

In [ ]:
# AJUSTE AQUI conforme o tempo de GPU disponível.
# As células com !python abaixo usam {EPOCHS}/{BATCH_SIZE}/{PATIENCE} entre chaves —
# é interpolação de variáveis Python do IPython/Colab, não precisa editar essas células,
# só rodar esta aqui de novo com valores diferentes se quiser mudar.
EPOCHS = 30
BATCH_SIZE = 8
PATIENCE = 5

### 4.1 Faster R-CNN

Se a sessão cair no meio do treino, rode de novo com
`--resume-from checkpoints/fasterrcnn_epoch<N>.pt` (troque `<N>` pela última época salva).

In [ ]:
!python src/train.py fasterrcnn --epochs {EPOCHS} --batch-size {BATCH_SIZE} --patience {PATIENCE}

### 4.2 SSD

In [ ]:
!python src/train.py ssd --epochs {EPOCHS} --batch-size {BATCH_SIZE} --patience {PATIENCE}

### 4.3 YOLOv8

Se a sessão cair no meio do treino, rode de novo com `--resume` no lugar de `--epochs`.

In [ ]:
!python src/train.py yolo --epochs {EPOCHS} --batch-size {BATCH_SIZE} --patience {PATIENCE}

## 5. Resultados

> INSTRUÇÃO (apague depois de escrever): rode as células abaixo (avaliação + gráficos) e
> depois escreva sua análise da tabela de mAP e das curvas de loss — qual modelo teve o
> melhor mAP@0.5 e mAP@0.5:0.95, algum modelo mostrou sinais de overfitting (val_loss subindo
> enquanto train_loss cai)?

In [ ]:
# Acha automaticamente o último checkpoint salvo de cada modelo (não assume que o treino
# completou todas as EPOCHS — o early stopping pode ter parado antes).
from pathlib import Path

def latest_checkpoint(model_name):
    ckpts = sorted(Path('checkpoints').glob(f'{model_name}_epoch*.pt'),
                   key=lambda p: int(p.stem.rsplit('epoch', 1)[-1]))
    return ckpts[-1]

fasterrcnn_ckpt = latest_checkpoint('fasterrcnn')
ssd_ckpt = latest_checkpoint('ssd')
print('Usando checkpoints:', fasterrcnn_ckpt, ssd_ckpt)

In [ ]:
# Avaliação (mAP via pycocotools)
!python src/eval.py fasterrcnn --checkpoint {fasterrcnn_ckpt} --batch-size {BATCH_SIZE}
!python src/eval.py ssd        --checkpoint {ssd_ckpt} --batch-size {BATCH_SIZE}
!python src/eval.py yolo       --checkpoint checkpoints/yolo/weights/best.pt

In [ ]:
# Gráficos: curva de loss de cada modelo + comparação de mAP
!python src/plotting.py loss fasterrcnn
!python src/plotting.py loss ssd
!python src/plotting.py loss yolo
!python src/plotting.py map --metrics-path logs/metrics.json

In [ ]:
from IPython.display import Image as IPImage, display
for path in ['logs/fasterrcnn_loss_curve.png', 'logs/ssd_loss_curve.png',
             'logs/yolo_loss_curve.png', 'logs/map_comparison.png']:
    display(IPImage(path))

> INSTRUÇÃO (apague depois de escrever): o checklist do trabalho pede também exemplos
> visuais de acertos/erros de cada modelo. A célula abaixo desenha as predições de um modelo
> em algumas imagens de teste — rode pros 3 modelos, escolha 2-3 exemplos bons e 1-2 ruins
> de cada, e comente o que observou.

In [ ]:
# Visualiza predições de um modelo em algumas imagens de teste (não faz parte de src/,
# é só uma célula de inspeção rápida pro notebook).
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def show_predictions(model_name, num_images=3, score_threshold=0.3):
    ann = json.load(open('data/coco/annotations/instances_test.json'))
    preds = json.load(open(f'logs/{model_name}_predictions.json'))
    id_to_name = {c['id']: c['name'] for c in ann['categories']}
    images_by_id = {img['id']: img for img in ann['images']}

    preds_by_image = {}
    for p in preds:
        if p['score'] >= score_threshold:
            preds_by_image.setdefault(p['image_id'], []).append(p)

    if not preds_by_image:
        print(f'{model_name}: nenhuma predição acima de score_threshold={score_threshold} '
              f'nas imagens de teste. Tente baixar o score_threshold.')
        return

    image_ids = list(preds_by_image.keys())[:num_images]
    fig, axes = plt.subplots(1, len(image_ids), figsize=(6 * len(image_ids), 6))
    if len(image_ids) == 1:
        axes = [axes]

    for ax, image_id in zip(axes, image_ids):
        img_info = images_by_id[image_id]
        img = Image.open(Path('data/coco/images/test') / img_info['file_name'])
        ax.imshow(img)
        for p in preds_by_image[image_id]:
            x, y, w, h = p['bbox']
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=2))
            ax.text(x, y, f"{id_to_name[p['category_id']]} {p['score']:.2f}", color='white',
                    fontsize=8, bbox=dict(facecolor='red', alpha=0.7, pad=1))
        ax.set_title(f'{model_name} — imagem {image_id}')
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'logs/{model_name}_sample_predictions.png')
    plt.show()

show_predictions('fasterrcnn')

## 6. Conclusão

> INSTRUÇÃO (apague esta linha depois de escrever): compare os 3 modelos considerando mAP,
> velocidade de treino/inferência observada e facilidade de uso, e conclua qual você
> recomendaria para um cenário real de catalogação de moda, justificando.